# Exact-Match Lyrics Lookup

This notebook looks up song lyrics using **artist**, **album**, and **song title** via the [LRCLIB API](https://lrclib.net/docs) (`GET /api/get`).

- **No API key required** and **zero external Python dependencies** (uses the Python standard library).
- **No guessing:** It queries LRCLIB's exact metadata endpoint and verifies that the returned artist, album, and track title match your input (case-insensitive, whitespace-normalized). If no exact match exists or no lyrics text is available, it reports that the lyrics were not found.

In [ ]:
from dataclasses import dataclass
import json
import re
from typing import Optional
import urllib.error
import urllib.parse
import urllib.request

LRCLIB_GET_URL = "https://lrclib.net/api/get"
USER_AGENT = "Jupyter-Lyrics-Lookup/1.0 (https://github.com/)"
REQUEST_TIMEOUT_SECONDS = 15

In [ ]:
@dataclass
class LyricsResult:
    found: bool
    status: str  # "found", "not_found", "instrumental", "no_lyrics_text", "error"
    artist: str
    album: str
    song_title: str
    plain_lyrics: Optional[str] = None
    synced_lyrics: Optional[str] = None
    matched_artist: Optional[str] = None
    matched_album: Optional[str] = None
    matched_track: Optional[str] = None
    track_id: Optional[int] = None
    message: Optional[str] = None


def _normalize(text: str) -> str:
    """Normalize whitespace and case for strict comparison without fuzzy guessing."""
    return re.sub(r"\s+", " ", text.strip()).casefold()


def lookup_lyrics(artist: str, album: str, song_title: str) -> LyricsResult:
    """
    Look up exact-match lyrics for (artist, album, song_title) via LRCLIB.
    Never falls back to fuzzy search or guessing.
    """
    artist_clean = artist.strip()
    album_clean = album.strip()
    title_clean = song_title.strip()

    if not artist_clean or not album_clean or not title_clean:
        return LyricsResult(
            found=False,
            status="error",
            artist=artist_clean,
            album=album_clean,
            song_title=title_clean,
            message="Artist, album, and song title must all be non-empty.",
        )

    params = urllib.parse.urlencode({
        "artist_name": artist_clean,
        "album_name": album_clean,
        "track_name": title_clean,
    })
    url = f"{LRCLIB_GET_URL}?{params}"
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})

    try:
        with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT_SECONDS) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            return LyricsResult(
                found=False,
                status="not_found",
                artist=artist_clean,
                album=album_clean,
                song_title=title_clean,
                message="No exact match found in LRCLIB.",
            )
        return LyricsResult(
            found=False,
            status="error",
            artist=artist_clean,
            album=album_clean,
            song_title=title_clean,
            message=f"HTTP error {exc.code}: {exc.reason}",
        )
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
        return LyricsResult(
            found=False,
            status="error",
            artist=artist_clean,
            album=album_clean,
            song_title=title_clean,
            message=f"Request failed: {exc}",
        )

    matched_artist = data.get("artistName") or ""
    matched_album = data.get("albumName") or ""
    matched_track = data.get("trackName") or ""

    # Reject near-matches so we never return lyrics for a different track or album.
    if (
        _normalize(matched_artist) != _normalize(artist_clean)
        or _normalize(matched_album) != _normalize(album_clean)
        or _normalize(matched_track) != _normalize(title_clean)
    ):
        return LyricsResult(
            found=False,
            status="not_found",
            artist=artist_clean,
            album=album_clean,
            song_title=title_clean,
            matched_artist=matched_artist,
            matched_album=matched_album,
            matched_track=matched_track,
            track_id=data.get("id"),
            message="Returned metadata did not strictly match input; rejected without guessing.",
        )

    if data.get("instrumental"):
        return LyricsResult(
            found=False,
            status="instrumental",
            artist=artist_clean,
            album=album_clean,
            song_title=title_clean,
            matched_artist=matched_artist,
            matched_album=matched_album,
            matched_track=matched_track,
            track_id=data.get("id"),
            message="Track is marked as instrumental.",
        )

    plain_lyrics = (data.get("plainLyrics") or "").strip() or None
    synced_lyrics = (data.get("syncedLyrics") or "").strip() or None

    if not plain_lyrics and not synced_lyrics:
        return LyricsResult(
            found=False,
            status="no_lyrics_text",
            artist=artist_clean,
            album=album_clean,
            song_title=title_clean,
            matched_artist=matched_artist,
            matched_album=matched_album,
            matched_track=matched_track,
            track_id=data.get("id"),
            message="Track matched, but no lyrics text is stored.",
        )

    return LyricsResult(
        found=True,
        status="found",
        artist=artist_clean,
        album=album_clean,
        song_title=title_clean,
        plain_lyrics=plain_lyrics,
        synced_lyrics=synced_lyrics,
        matched_artist=matched_artist,
        matched_album=matched_album,
        matched_track=matched_track,
        track_id=data.get("id"),
    )


def display_lyrics(result: LyricsResult) -> None:
    """Print a clean summary and lyrics (if found)."""
    print(f"{result.artist} — {result.song_title} ({result.album})")
    print(f"Status: {result.status}")
    if not result.found:
        if result.message:
            print(f"Details: {result.message}")
        return
    print("-" * 60)
    if result.plain_lyrics:
        print(result.plain_lyrics)
    elif result.synced_lyrics:
        print("[Plain lyrics unavailable; synced LRC lyrics below]")
        print(result.synced_lyrics)

In [ ]:
# Interactive single-track lookup
artist_input = input("Artist: ")
album_input = input("Album: ")
song_input = input("Song title: ")

result = lookup_lyrics(artist_input, album_input, song_input)
display_lyrics(result)

In [ ]:
# Programmatic example (exact match vs. not found without guessing)
examples = [
    ("Fleetwood Mac", "Rumours", "The Chain"),
    ("Fleetwood Mac", "Nonexistent Album", "The Chain"),
]

for artist, album, title in examples:
    res = lookup_lyrics(artist, album, title)
    print(f"Query: {artist} | {album} | {title} -> found={res.found}, status={res.status}")
    if res.found and res.plain_lyrics:
        preview = "\n".join(res.plain_lyrics.splitlines()[:4])
        print(f"Preview:\n{preview}\n...")
    elif res.message:
        print(f"Message: {res.message}")
    print()